# GPT Action 库：Confluence

## 简介

本页面为开发者构建特定应用的 GPT Action 提供说明与指南。在继续之前，请先熟悉以下内容： 
- [GPT Actions 简介](https://platform.openai.com/docs/actions)
- [GPT Actions 库简介](https://platform.openai.com/docs/actions/actions-library)
- [从零构建 GPT Action 的示例](https://platform.openai.com/docs/actions/getting-started)

这个 GPT Action 概述了如何连接到 **Confluence**——Atlassian 的协作与文档平台。该 Action 接收用户的问题，扫描相关的 Confluence 空间和页面以收集所需信息，然后组织回答来解答用户的问题。本 cookbook 不涉及直接从 ChatGPT 更新 Confluence 中的内容，但借助额外的 Actions 和权限范围，这在技术上是可行的。

### 价值与业务用例示例

**价值**

用户现在可以利用 ChatGPT 的自然语言能力直接连接到 Confluence，与组织的知识库进行无缝交互。

**用例示例**
- **知识工作者**：轻松从 Confluence 页面和空间中检索信息，以回答问题或为报告和演示收集细节。
- **项目经理**：快速访问存储在 Confluence 中的项目文档和更新，无需手动翻阅页面。
- **客户支持团队**：从 Confluence 知识库中提取相关信息，准确及时地回复客户咨询。
- **所有用户**：更全面地了解全公司范围的文档、政策和流程，促进协作与知识共享。

## 应用信息

### 应用关键链接

开始之前，请查看该应用的以下链接：
- 应用网站：https://developer.atlassian.com/console/myapps/ 
- 应用 API 文档：https://developer.atlassian.com/cloud/confluence/rest/v2/intro/#about

### 应用前置条件

开始之前，请确保在你的应用环境中完成以下步骤：
- 确保你有权限在 Atlassian Developer Portal 中创建 App
- 确定你希望 GPT 执行哪些交互（搜索、读取、编辑等）

## ChatGPT 步骤

### 自定义 GPT 指令 

创建自定义 GPT 后，将下面的文本复制到 Instructions 面板中。有疑问？请查看[入门示例](https://platform.openai.com/docs/actions/getting-started)，更详细地了解这一步的工作方式。

In [ ]:
You are a "Confluence Savant", equipped with the ability to search our company's Product Wiki in Confluence to answer product-related questions.

You must ALWAYS perform the "getAccessibleResources" Action first to get the "cloudid" value you will need in subsequent Actions.

Your job is to provide accurate and detailed responses by retrieving information from the Product Wiki. Your responses should be clear, concise, and directly address the question asked. You have the capability to execute an action named "performConfluenceSearch" that allows you to search for content within our Confluence Product Wiki using specific terms or phrases related to the user's question.

    - When you receive a query about product information, use the "performConfluenceSearch" action to retrieve relevant content from the Product Wiki. Formulate your search query based on the user's question, using specific keywords or phrases to find the most pertinent information.
    - Once you receive the search results, review the content to ensure it matches the user's query. If necessary, refine your search query to retrieve more accurate results.
    - Provide a response that synthesizes the information from the Product Wiki, clearly answering the user's question. Your response should be easy to understand and directly related to the query.
    - If the query is complex or requires clarification, ask follow-up questions to the user to refine your understanding and improve the accuracy of your search.
    - If the information needed to answer the question is not available in the Product Wiki, inform the user and guide them to where they might find the answer, such as contacting a specific department or person in the company.

    Here is an example of how you might respond to a query:

    User: "What are the latest features of our XYZ product?"
    You: "The latest features of the XYZ product, as detailed in our Product Wiki, include [feature 1], [feature 2], and [feature 3]. These features were added in the recent update to enhance [specific functionalities]. For more detailed information, you can refer to the Product Wiki page [link to the specific Confluence page]."

Remember, your goal is to provide helpful, accurate, and relevant information to the user's query by effectively leveraging the Confluence Product Wiki.


### OpenAPI Schema 

创建自定义 GPT 后，将下面的文本复制到 Actions 面板中。有疑问？请查看[入门示例](https://platform.openai.com/docs/actions/getting-started)，更详细地了解这一步的工作方式。

In [ ]:
openapi: 3.1.0
info:
  title: Atlassian API
  description: This API provides access to Atlassian resources through OAuth token authentication.
  version: 1.0.0
servers:
  - url: https://api.atlassian.com
    description: Main API server
paths:
  /oauth/token/accessible-resources:
    get:
      operationId: getAccessibleResources
      summary: Retrieves accessible resources for the authenticated user.
      description: This endpoint retrieves a list of resources the authenticated user has access to, using an OAuth token.
      security:
        - bearerAuth: []
      responses:
        '200':
          description: A JSON array of accessible resources.
          content:
            application/json:
              schema: 
                $ref: '#/components/schemas/ResourceArray'
  /ex/confluence/{cloudid}/wiki/rest/api/search:
    get:
      operationId: performConfluenceSearch
      summary: Performs a search in Confluence based on a query.
      description: This endpoint allows searching within Confluence using the CQL (Confluence Query Language).
      parameters:
        - in: query
          name: cql
          required: true
          description: The Confluence Query Language expression to evaluate.
          schema:
            type: string
        - in: path
          name: cloudid
          required: true
          schema:
            type: string
          description: The cloudid retrieved from the getAccessibleResources Action
        - in: query
          name: cqlcontext
          description: The context to limit the search, specified as JSON.
          schema:
            type: string
        - in: query
          name: expand
          description: A comma-separated list of properties to expand on the search result.
          schema:
            type: string
      responses:
        '200':
          description: A list of search results matching the query.
          content:
            application/json:
              schema:
                $ref: '#/components/schemas/SearchResults'
components:
  securitySchemes:
    bearerAuth:
      type: http
      scheme: bearer
      bearerFormat: JWT
  schemas:
    ResourceArray:
      type: array
      items:
        $ref: '#/components/schemas/Resource'
    Resource:
      type: object
      required:
        - id
        - name
        - type
      properties:
        id:
          type: string
          description: The unique identifier for the resource.
        name:
          type: string
          description: The name of the resource.
        type:
          type: string
          description: The type of the resource.
    SearchResults:
      type: object
      properties:
        results:
          type: array
          items:
            $ref: '#/components/schemas/SearchResult'
    SearchResult:
      type: object
      properties:
        id:
          type: string
          description: The unique identifier of the content.
        title:
          type: string
          description: The title of the content.
        type:
          type: string
          description: The type of the content (e.g., page, blog post).
        space:
          type: object
          properties:
            id:
              type: string
              description: The space ID where the content is located.
            name:
              type: string
              description: The name of the space.

## 身份验证说明

以下是与该第三方应用设置身份验证的说明。有疑问？请查看[入门示例](https://platform.openai.com/docs/actions/getting-started)，更详细地了解这一步的工作方式。

### Action 前置步骤

在 ChatGPT 中设置身份验证之前，请确保在 Atlassian Developer portal 中完成以下步骤来创建你的 Confluence 应用：

1. 选择 Create 下拉菜单
2. 选择 OAuth 2.0 integration
3. 填写名称，同意条款，然后点击 Create
4. 在左侧菜单中选择 "Distribution"，然后点击 “Edit”
5. 将单选按钮切换为 "Sharing"
6. 填写必填字段并保存更改（Save Changes）
7. 在左侧菜单中选择 "Permissions"
8. 添加你希望包含的权限范围（例如 User identity API 和 Confluence API，以便应用能够知道用户有权访问哪些内容并从 Confluence 获取数据）
9. 在左侧菜单中选择 "Authorization"
10. 在 OAuth 2.0 这一行的 Action 下点击 "Add"
11. 输入来自 GPT 的回调 URL（注意：你可能需要先填一个占位符，待在 GPT 中创建好 Action 和 OAuth、获得最终回调 URL 后再回来修改）
12. 在左侧菜单中选择 "Settings"
13. 复制你的 Client ID 和 Secret，用于在 GPT 中进行 OAuth 设置


![confluence_gpt.png](../../../images/confluence_gpt.png)

### 在 ChatGPT 中

在 ChatGPT 中，点击 "Authentication" 并选择 **"OAuth"**。输入以下信息。 

- **Client ID**：使用上述步骤中的 Client ID 
- **Client Secret**：使用上述步骤中的 Client Secret
- **Authorization URL**：https://auth.atlassian.com/authorize
- **Token URL**：https://auth.atlassian.com/oauth/token 
- **Scope**：read:confluence-content.all search:confluence
- **Token**：Default (POST)

### Action 后续步骤

在 ChatGPT 中设置好身份验证后，请在应用中按照以下步骤完成 Action 的配置。 

- 从 GPT Action 中复制回调 URL
- 在 “Authorized redirect URIs”（见上方截图）中添加你的回调 URL 


### 常见问题与故障排除

- *回调 URL 错误：* 如果你在 ChatGPT 中遇到回调 URL 错误，请仔细查看上方截图。你需要将回调 URL 直接添加到 Confluence 应用中，Action 才能正确完成身份验证
- *Schema 调用了错误的项目或数据集：* 如果 ChatGPT 调用了错误的项目或数据集，可以考虑更新指令，使其更加明确：(a) 应调用哪个项目/数据集，或 (b) 要求用户在运行查询之前提供这些确切细节
- *Action 循环调用：* 你可能没有为应用授予实现预期目的所需的权限范围/权限

*有希望我们优先支持的集成吗？我们的集成中有错误吗？请在我们的 GitHub 上提交 PR 或 issue，我们会查看。*